# 06 - SHAP: Interpretabilidad del modelo

**Proyecto:** Prediccion de desercion estudiantil  
**Modelo evaluado:** LogReg L2 (campeon, ADR-08)  
**Integrantes:** Jorge Garcia, Jose Villanueva, Aldair Rivas  

Objetivos:
1. Construir un `LinearExplainer` sobre el conjunto de entrenamiento.
2. Calcular valores SHAP globales (todos los estudiantes de train).
3. Generar plots globales: summary bar y beeswarm.
4. Generar plots locales: waterfall para los 3 estudiantes de mayor riesgo.
5. Registrar figuras en MLflow (experimento `dropout_tf1`, run `shap_analysis`).

## 1. Setup

In [5]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Ruta al proyecto y al codigo fuente
ROOT = Path("../").resolve()
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from config import (
    TRAIN_PATH, MODELS_DIR, FIGURES_DIR, TARGET, SEED
)
from mlflow_utils import setup_tracking
import shap_utils as su

print("[OK] Imports listos")
print(f"ROOT: {ROOT}")

[OK] Imports listos
ROOT: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining


## 2. Carga del modelo y datos de entrenamiento

In [6]:
# Cargar modelo (suprimir advertencia de version de sklearn)
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=".*InconsistentVersionWarning.*",
        category=UserWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message=".*Trying to unpickle.*",
        category=UserWarning,
    )
    pipeline = joblib.load(MODELS_DIR / "dropout_logreg_v0.1-tp1.joblib")

print("[OK] Pipeline cargado")
print("Pasos:", list(pipeline.named_steps.keys()))

[OK] Pipeline cargado
Pasos: ['prep', 'model']


In [7]:
# Cargar datos de entrenamiento
train_df = pd.read_csv(TRAIN_PATH)
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

print(f"[OK] Train cargado: {X_train.shape[0]} filas, {X_train.shape[1]} columnas")
print(f"     Dropout rate: {y_train.mean():.3f}")

[OK] Train cargado: 3539 filas, 32 columnas
     Dropout rate: 0.321


In [8]:
# Extraer nombres de features tras el preprocesador
feature_names = su.get_feature_names(pipeline)
print(f"[OK] Features tras preprocesador: {len(feature_names)}")
print("Primeras 10:", feature_names[:10])

[OK] Features tras preprocesador: 102
Primeras 10: ['Application order', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Unemployment rate', 'Inflation rate', 'GDP', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)']


## 3. Construccion del LinearExplainer

Se usa una muestra aleatoria de 500 filas como distribucion de fondo para el masker.

In [9]:
rng = np.random.default_rng(SEED)
bg_idx = rng.choice(len(X_train), size=min(500, len(X_train)), replace=False)
X_background = X_train.iloc[bg_idx].reset_index(drop=True)

print(f"Muestra de fondo: {len(X_background)} filas")

explainer, X_bg_t = su.build_explainer(pipeline, X_background)

print("[OK] LinearExplainer construido")
print(f"     Valor esperado (E[f(X)]): {explainer.expected_value:.4f}")

Muestra de fondo: 500 filas
[OK] LinearExplainer construido
     Valor esperado (E[f(X)]): -1.1102


## 4. Calculo de valores SHAP sobre el conjunto de entrenamiento completo

In [10]:
print("Calculando SHAP values sobre train completo...")
shap_vals, X_train_t = su.compute_shap_values(explainer, pipeline, X_train)

print(f"[OK] SHAP values calculados")
print(f"     Shape: {shap_vals.shape}  (muestras x features)")
print(f"     Valor SHAP absoluto medio: {np.abs(shap_vals).mean():.4f}")

Calculando SHAP values sobre train completo...
[OK] SHAP values calculados
     Shape: (3539, 102)  (muestras x features)
     Valor SHAP absoluto medio: 0.0912


## 5. Plots globales

In [11]:
# 5a. Summary bar (importancia media |SHAP|)
out_bar = FIGURES_DIR / "09_shap_summary_bar.png"
su.plot_summary_bar(shap_vals, feature_names, out_path=out_bar, top_n=20)

[OK] Guardado: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining\reports\figures\09_shap_summary_bar.png


In [12]:
# 5b. Beeswarm (distribucion de valores SHAP por feature)
out_bee = FIGURES_DIR / "10_shap_beeswarm.png"
su.plot_beeswarm(shap_vals, X_train_t, feature_names, out_path=out_bee, top_n=20)

[OK] Guardado: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining\reports\figures\10_shap_beeswarm.png


## 6. Plots locales: waterfall para los 3 estudiantes de mayor riesgo

Se toman los estudiantes de train con mayor probabilidad predicha de desercion.

In [13]:
# Probabilidades de dropout predichas para todo train
probs = pipeline.predict_proba(X_train)[:, 1]

# Top 3 de mayor riesgo
top3_idx = np.argsort(probs)[::-1][:3]

print("Top 3 estudiantes de mayor riesgo en train:")
for rank, i in enumerate(top3_idx, start=1):
    real = int(y_train.iloc[i])
    print(f"  Rank {rank}: fila {i}, P(dropout)={probs[i]:.3f}, real={'Dropout' if real else 'No Dropout'}")

Top 3 estudiantes de mayor riesgo en train:
  Rank 1: fila 1835, P(dropout)=1.000, real=Dropout
  Rank 2: fila 1798, P(dropout)=1.000, real=Dropout
  Rank 3: fila 1219, P(dropout)=1.000, real=Dropout


In [14]:
expected_value = float(explainer.expected_value)

for rank, i in enumerate(top3_idx, start=1):
    out_wf = FIGURES_DIR / f"11_shap_waterfall_rank{rank}.png"
    su.plot_waterfall_single(
        shap_vals_single=shap_vals[i],
        X_single=X_train_t[i],
        feature_names=feature_names,
        expected_value=expected_value,
        out_path=out_wf,
        label=f"Estudiante rank {rank} - P(dropout)={probs[i]:.3f}",
    )

[OK] Guardado: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining\reports\figures\11_shap_waterfall_rank1.png
[OK] Guardado: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining\reports\figures\11_shap_waterfall_rank2.png
[OK] Guardado: C:\Users\rivas\Desktop\2026\2026-2\Data Mining Tools\TP1_DataMining\reports\figures\11_shap_waterfall_rank3.png


## 7. Registro de figuras en MLflow

In [15]:
import mlflow

setup_tracking()  # sqlite:///mlflow.db
mlflow.set_experiment("dropout_tf1")

with mlflow.start_run(run_name="shap_analysis") as run:
    # Parametros de contexto
    mlflow.log_params({
        "model": "LogReg_L2_campeon",
        "n_train": len(X_train),
        "n_background": len(X_background),
        "n_features": len(feature_names),
        "shap_explainer": "LinearExplainer",
    })

    # Metricas resumen
    mean_abs_shap = float(np.abs(shap_vals).mean())
    mlflow.log_metrics({
        "mean_abs_shap": mean_abs_shap,
        "expected_value": expected_value,
    })

    # Figuras globales
    mlflow.log_artifact(str(FIGURES_DIR / "09_shap_summary_bar.png"), artifact_path="shap")
    mlflow.log_artifact(str(FIGURES_DIR / "10_shap_beeswarm.png"), artifact_path="shap")

    # Figuras locales
    for rank in range(1, 4):
        fp = FIGURES_DIR / f"11_shap_waterfall_rank{rank}.png"
        if fp.exists():
            mlflow.log_artifact(str(fp), artifact_path="shap")

    print(f"[OK] Run registrado en MLflow: {run.info.run_id}")
    print(f"     mean |SHAP|: {mean_abs_shap:.4f}")
    print(f"     E[f(X)]:     {expected_value:.4f}")

[OK] Run registrado en MLflow: c5ada757cf9a449692d87588980a0202
     mean |SHAP|: 0.0912
     E[f(X)]:     -1.1102


## 8. Resumen de hallazgos

Ejecutar esta celda despues de ver los plots para registrar las observaciones.

In [16]:
# Top features por importancia media |SHAP|
mean_abs = np.abs(shap_vals).mean(axis=0)
top_idx = np.argsort(mean_abs)[::-1][:10]

print("Top 10 features por importancia SHAP global:")
print(f"{'Rank':<5} {'Feature':<55} {'Mean |SHAP|':>12}")
print("-" * 74)
for rank, i in enumerate(top_idx, start=1):
    print(f"{rank:<5} {feature_names[i]:<55} {mean_abs[i]:>12.4f}")

Top 10 features por importancia SHAP global:
Rank  Feature                                                  Mean |SHAP|
--------------------------------------------------------------------------
1     Curricular units 1st sem (approved)                           1.7771
2     Curricular units 1st sem (enrolled)                           0.5827
3     Tuition fees up to date                                       0.5349
4     Curricular units 1st sem (credited)                           0.3640
5     sin_evaluaciones_1s                                           0.3564
6     Unemployment rate                                             0.3288
7     Course_171                                                    0.2315
8     Age at enrollment                                             0.2291
9     Gender                                                        0.2287
10    Mother's occupation_9                                         0.2106
